In [1]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 1.1 MB/s  0:01:49 eta 0:00:010:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 MB 1.2 MB/s  0:06:15 eta 0:00:010:00:06m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]━━━ 1/2 [xgboost]


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, json, gc
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, accuracy_score,
                              f1_score, precision_score, recall_score,
                              precision_recall_curve, average_precision_score)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
import joblib

np.random.seed(42)
print('ALL IMPORTS DONE!')

In [ ]:
def parse_vcf(filepath):
    records = []
    header  = None
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('##'):
                continue
            elif line.startswith('#CHROM'):
                header = line.lstrip('#').split('\t')
            else:
                if header:
                    fields = line.split('\t')
                    records.append(fields)
    df = pd.DataFrame(records, columns=header)
    print(f"Loaded {len(df)} total variants")
    return df

df_full = parse_vcf("clinvar_20260302.vcf")

In [ ]:
print("\n🔍 Checking available fields in your ClinVar VCF...")

check_fields = {
    'Allele Freq (AF_ESP)'   : 'AF_ESP',
    'Allele Freq (AF_EXAC)'  : 'AF_EXAC',
    'Allele Freq (AF_TGP)'   : 'AF_TGP',
    'PolyPhen'               : 'polyphen',
    'SIFT'                   : 'sift',
    'CADD'                   : 'CADD',
    'PhyloP'                 : 'phylop',
    'PhastCons'              : 'phastcons',
    'Homozygous'             : 'CLNHGVS',
    'Molecular Consequence'  : 'MC=',
    'Functional Class'       : 'GENEINFO',
    'dbSNP RS'               : 'RS=',
    'Origin'                 : 'CLNORIGIN',
    'Review Status'          : 'CLNREVSTAT',
    'Allele Count'           : 'AC=',
}

sample_info = df_full['INFO'].head(10000)
print(f"\n{'Field':<30} {'Found?':>10} {'Example Value'}")
print("-" * 70)
for label, tag in check_fields.items():
    mask    = sample_info.str.contains(tag, case=False, na=False)
    found   = mask.sum()
    example = ''
    if found > 0:
        match = sample_info[mask].iloc[0]
        import re
        ex = re.findall(rf'{tag}[=:]?([^;,\s]{{0,20}})', match, re.IGNORECASE)
        example = ex[0] if ex else 'present'
    status = f'{found}/10000' if found > 0 else 'Not found'
    print(f"{label:<30} {status:<20} {example}")

In [ ]:
def filter_missense(df):
    print(f"\nBefore filtering : {len(df)} variants")
    missense_mask = df['INFO'].str.contains('missense', case=False, na=False)
    df_filtered   = df[missense_mask].copy().reset_index(drop=True)
    print(f"After filtering  : {len(df_filtered)} missense variants")
    return df_filtered

df_missense = filter_missense(df_full)


In [ ]:
sampled_indices = df_missense.sample(n=10000, random_state=42).index
df_main         = df_missense.loc[sampled_indices].copy().reset_index(drop=True)

df_remaining    = df_missense[~df_missense.index.isin(sampled_indices)].copy()
df_remaining    = df_remaining.reset_index(drop=True)
df_remaining.to_csv('remaining_unseen.csv', index=False)

print(f"Training pool    : {len(df_main)} samples")
print(f" Remaining unseen : {len(df_remaining)} variants → remaining_unseen.csv")

del df_full
del df_missense
del df_remaining
gc.collect()
print("  Full data cleared from memory!")

In [ ]:
def clean_data(df):
    print(f"Before cleaning: {df.shape}")
    df = df.drop_duplicates()
    df = df.dropna(how='all')
    df = df.reset_index(drop=True)
    print(f"After cleaning : {df.shape}")
    return df

df_main = clean_data(df_main)

In [ ]:
def extract_features(df):
    features = pd.DataFrame()
    info     = df['INFO']
    #CELL 1 — Positional Features
features = pd.DataFrame()
info     = df_main['INFO']

features['CHROM']         = pd.factorize(df_main['CHROM'].astype(str))[0]
features['POS']           = pd.to_numeric(df_main['POS'], errors='coerce')
features['REF_len']       = df_main['REF'].str.len()
features['REF_encoded']   = pd.factorize(df_main['REF'].astype(str))[0]
features['ALT_len']       = df_main['ALT'].str.len()
features['ALT_encoded']   = pd.factorize(df_main['ALT'].astype(str))[0]
features['is_snp']        = ((features['REF_len']==1) &
                              (features['ALT_len']==1)).astype(int)
features['is_indel']      = (features['REF_len'] !=
                              features['ALT_len']).astype(int)
features['mutation_size'] = (features['ALT_len'] -
                              features['REF_len']).abs()

print(f"Cell 1 done — Positional features: {features.shape[1]} cols")
print(f"   Columns: {features.columns.tolist()}")

In [ ]:
#CELL 2 — Allele Frequency
features['AF_ESP']  = pd.to_numeric(
    info.str.extract(r'AF_ESP=([^;]+)')[0],  errors='coerce')
features['AF_EXAC'] = pd.to_numeric(
    info.str.extract(r'AF_EXAC=([^;]+)')[0], errors='coerce')
features['AF_TGP']  = pd.to_numeric(
    info.str.extract(r'AF_TGP=([^;]+)')[0],  errors='coerce')

features['AF_combined'] = features['AF_ESP'].combine_first(
                          features['AF_EXAC']).combine_first(
                          features['AF_TGP'])

print(f" Cell 2 done — Allele frequency features added")
print(f"   AF_ESP    non-null: {features['AF_ESP'].notna().sum()}")
print(f"   AF_EXAC   non-null: {features['AF_EXAC'].notna().sum()}")
print(f"   AF_TGP    non-null: {features['AF_TGP'].notna().sum()}")
print(f"   AF_combined non-null: {features['AF_combined'].notna().sum()}")

In [ ]:
#CELL 3 — PolyPhen Score 
features['polyphen_score'] = pd.to_numeric(
    info.str.extract(r'[Pp]oly[Pp]hen[_=]([0-9.]+)')[0],
    errors='coerce')

polyphen_raw = info.str.extract(
    r'[Pp]oly[Pp]hen=([^;,\s]+)')[0].fillna('unknown')
polyphen_map = {
    'probably_damaging' : 2,
    'possibly_damaging' : 1,
    'benign'            : 0,
    'unknown'           : -1
}
features['polyphen_cat'] = polyphen_raw.map(
    lambda x: next((v for k,v in polyphen_map.items()
                    if k in x.lower()), -1))

dist = features['polyphen_cat'].value_counts().to_string()
print(f" Cell 3 done — PolyPhen features added")
print(f"   polyphen_score non-null : {features['polyphen_score'].notna().sum()}")
print(f"   polyphen_cat distribution:")
print(dist)

In [ ]:
# CELL 4 — SIFT Score 
features['sift_score'] = pd.to_numeric(
    info.str.extract(r'[Ss][Ii][Ff][Tt][_=]([0-9.]+)')[0],
    errors='coerce')

sift_raw = info.str.extract(
    r'[Ss][Ii][Ff][Tt]=([^;,\s]+)')[0].fillna('unknown')
sift_map = {
    'deleterious' : 1,
    'tolerated'   : 0,
    'unknown'     : -1
}
features['sift_cat'] = sift_raw.map(
    lambda x: next((v for k,v in sift_map.items()
                    if k in x.lower()), -1))

print(f"  Cell 4 done — SIFT features added")
print(f"   sift_score non-null : {features['sift_score'].notna().sum()}")
print(f"   sift_cat distribution:\n{features['sift_cat'].value_counts().to_string()}")

In [ ]:
# CELL 5 — CADD, PhyloP, PhastCons Scores 
features['cadd_score'] = pd.to_numeric(
    info.str.extract(r'CADD[_=]([0-9.]+)')[0],
    errors='coerce')

features['phylop_score'] = pd.to_numeric(
    info.str.extract(r'[Pp]hylo[Pp][_=]([0-9.-]+)')[0],
    errors='coerce')

features['phastcons_score'] = pd.to_numeric(
    info.str.extract(r'[Pp]hast[Cc]ons[_=]([0-9.]+)')[0],
    errors='coerce')

print(f"    Cell 5 done — Conservation scores added")
print(f"   cadd_score    non-null : {features['cadd_score'].notna().sum()}")
print(f"   phylop_score  non-null : {features['phylop_score'].notna().sum()}")
print(f"   phastcons     non-null : {features['phastcons_score'].notna().sum()}")

In [ ]:
# CELL 6 — Homozygous Flags 
features['is_homozygous'] = info.str.contains(
    'CLNORIGIN=4', na=False).astype(int)

features['has_homozygous_flag'] = info.str.contains(
    'hom', case=False, na=False).astype(int)

print(f"  Cell 6 done — Homozygous flags added")
print(f"   is_homozygous       : {features['is_homozygous'].sum()}")
print(f"   has_homozygous_flag : {features['has_homozygous_flag'].sum()}")

In [ ]:
#CELL 7 — Review Status 
features['is_reviewed']          = info.str.contains(
    'CLNREVSTAT=reviewed_by_expert_panel', na=False).astype(int)
features['is_criteria_provided'] = info.str.contains(
    'CLNREVSTAT=criteria_provided', na=False).astype(int)
features['is_no_criteria']       = info.str.contains(
    'CLNREVSTAT=no_criteria', na=False).astype(int)

print(f" Cell 7 done — Review status flags added")
print(f"   is_reviewed          : {features['is_reviewed'].sum()}")
print(f"   is_criteria_provided : {features['is_criteria_provided'].sum()}")
print(f"   is_no_criteria       : {features['is_no_criteria'].sum()}")

In [ ]:
#  CELL 8 — Gene Info 
features['has_gene_info'] = info.str.contains(
    'GENEINFO=', na=False).astype(int)

gene_info = info.str.extract(r'GENEINFO=([^:;]+)')[0]
features['gene_encoded'] = pd.factorize(
    gene_info.fillna('Unknown'))[0]

print(f"  Cell 8 done — Gene info features added")
print(f"   has_gene_info : {features['has_gene_info'].sum()}")
print(f"   unique genes  : {gene_info.nunique()}")

In [ ]:
# CELL 9 — Allele Count & RS ID 
features['allele_count'] = pd.to_numeric(
    info.str.extract(r'AC=([^;]+)')[0], errors='coerce')

features['has_rs_id'] = info.str.contains(
    r'RS=\d', na=False).astype(int)

print(f" Cell 9 done — Allele count & RS ID added")
print(f"   allele_count non-null : {features['allele_count'].notna().sum()}")
print(f"   has_rs_id             : {features['has_rs_id'].sum()}")

In [ ]:
#  CELL 10 — Variant Quality Tags
for tag in ['DP', 'MQ', 'FS', 'SOR', 'QD']:
    features[tag] = pd.to_numeric(
        info.str.extract(rf'{tag}=([^;]+)')[0],
        errors='coerce')

print(f" Cell 10 done — Quality tags added")
for tag in ['DP', 'MQ', 'FS', 'SOR', 'QD']:
    print(f"   {tag:<5} non-null : {features[tag].notna().sum()}")

In [ ]:
# CELL 11 — Fill Missing Values 
before = features.isnull().sum().sum()
features = features.fillna(features.median(numeric_only=True))
after  = features.isnull().sum().sum()

print(f"  Cell 11 done — Missing values filled")
print(f"   Nulls before : {before}")
print(f"   Nulls after  : {after}")

In [ ]:
# CELL 12 — Final Summary 
X = features.copy()

score_cols = ['AF_combined', 'polyphen_score', 'sift_score',
              'cadd_score', 'phylop_score', 'phastcons_score']

print("=" * 55)
print("       FEATURE EXTRACTION COMPLETE")
print("=" * 55)
print(f"\n Score availability in your data:")
for col in score_cols:
    non_zero = (X[col] != 0).sum()
    pct      = non_zero / len(X) * 100
    status   = 'Right' if pct > 1 else '  sparse'
    print(f"   {col:<20} : {non_zero:>8} variants ({pct:.1f}%) {status}")

print(f"\n Total features extracted : {X.shape[1]}")
print(f"   Feature matrix shape     : {X.shape}")
print(f"\n All features:")
for i, col in enumerate(X.columns.tolist()):
    print(f"   {i+1:>2}. {col}")

In [ ]:
def prepare_labels(df, X):
    clnsig    = df['INFO'].str.extract(r'CLNSIG=([^;]+)')[0]
    label_map = {
        'Pathogenic'        : 1,
        'Likely_pathogenic' : 1,
        'Benign'            : 0,
        'Likely_benign'     : 0
    }
    y    = clnsig.map(label_map)
    mask = y.notna()

    X_clean = X[mask].reset_index(drop=True)
    y_clean = y[mask].astype(int).reset_index(drop=True)

    print("=" * 40)
    print("        LABEL REPORT")
    print("=" * 40)
    print(f"  Total labeled  : {len(y_clean)}")
    print(f"  Pathogenic (1) : {y_clean.sum()}")
    print(f"  Benign     (0) : {(y_clean==0).sum()}")
    print("=" * 40)
    return X_clean, y_clean

X, y = prepare_labels(df_main, X)

feature_names = X.columns.tolist()
with open('feature_names.json', 'w') as f:
    json.dump(feature_names, f)
print(f" Feature names saved → feature_names.json")

In [ ]:
total            = len(y)
benign_count     = (y==0).sum()
pathogenic_count = (y==1).sum()

print(f"\n  Benign      (0) : {benign_count} ({benign_count/total*100:.1f}%)")
print(f"  Pathogenic  (1) : {pathogenic_count} ({pathogenic_count/total*100:.1f}%)")
print(f"  Imbalance Ratio : {benign_count/pathogenic_count:.2f}:1")

classes       = np.array([0, 1])
weights       = compute_class_weight(class_weight='balanced',
                                     classes=classes, y=y)
class_weights = {0: weights[0], 1: weights[1]}
print(f"\n  Benign weight     : {weights[0]:.4f}")
print(f"  Pathogenic weight : {weights[1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].pie([benign_count, pathogenic_count],
            labels=['Benign', 'Pathogenic'],
            colors=['#4CAF50', '#F44336'],
            autopct='%1.1f%%', startangle=90,
            explode=(0.05, 0.05), shadow=True)
axes[0].set_title('Class Distribution')
bars = axes[1].bar(['Benign', 'Pathogenic'],
                   [benign_count, pathogenic_count],
                   color=['#4CAF50', '#F44336'], edgecolor='black')
axes[1].set_ylabel('Count')
axes[1].set_title('Class Count')
axes[1].grid(axis='y')
for bar, count in zip(bars, [benign_count, pathogenic_count]):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 10,
                 str(count), ha='center', fontweight='bold')
plt.suptitle('Class Imbalance Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp)

print("X_train, X_val, X_test, y_train, y_val, y_test are ready")

In [ ]:
import os
import numpy as np

JUPYTER_DIR = "ml_project_synaptix"
DESKTOP_DIR = os.path.join(os.path.expanduser("~"),
                           "Desktop", "ml_project_synaptix")

os.makedirs(JUPYTER_DIR, exist_ok=True)
os.makedirs(DESKTOP_DIR, exist_ok=True)

splits = {
    'X_train.npy': X_train,
    'X_val.npy':   X_val,
    'X_test.npy':  X_test,
    'y_train.npy': y_train,
    'y_val.npy':   y_val,
    'y_test.npy':  y_test,
}

for filename, data in splits.items():
    np.save(os.path.join(JUPYTER_DIR, filename), data)
    np.save(os.path.join(DESKTOP_DIR, filename), data)

print("=" * 50)
print("   SPLITS SAVED IN BOTH LOCATIONS")
print("=" * 50)
for f in os.listdir(JUPYTER_DIR):
    size = os.path.getsize(os.path.join(JUPYTER_DIR, f))
    print(f"   {f:<25} {size/1024:.1f} KB")

In [ ]:
pipelines = {
    'Random Forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   RandomForestClassifier(
                        n_estimators=200,
                        class_weight=class_weights,
                        random_state=42,
                        n_jobs=-1))
    ]),
    'XGBoost': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   XGBClassifier(
                        n_estimators=200,
                        learning_rate=0.05,
                        max_depth=6,
                        eval_metric='mlogloss',
                        random_state=42))
    ]),
    'SVM': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   SVC(
                        kernel='rbf',
                        class_weight=class_weights,
                        probability=True,
                        random_state=42))
    ])
}

results = {}
print("  Training All Models...\n")
for name, pipe in pipelines.items():
    print(f"  Training {name}...")
    pipe.fit(X_train, y_train)

    y_val_pred  = pipe.predict(X_val)
    y_val_prob  = pipe.predict_proba(X_val)[:, 1]
    val_acc     = accuracy_score(y_val, y_val_pred)
    val_f1      = f1_score(y_val, y_val_pred, average='weighted')
    val_auc     = roc_auc_score(y_val, y_val_prob)
    val_recall  = recall_score(y_val, y_val_pred, average='weighted')

    y_test_pred = pipe.predict(X_test)
    y_test_prob = pipe.predict_proba(X_test)[:, 1]
    test_acc    = accuracy_score(y_test, y_test_pred)
    test_f1     = f1_score(y_test, y_test_pred, average='weighted')
    test_auc    = roc_auc_score(y_test, y_test_prob)
    test_recall = recall_score(y_test, y_test_pred, average='weighted')

    results[name] = {
        'pipeline'   : pipe,
        'val_acc'    : val_acc,  'val_f1'    : val_f1,
        'val_auc'    : val_auc,  'val_recall': val_recall,
        'test_acc'   : test_acc, 'test_f1'   : test_f1,
        'test_auc'   : test_auc, 'test_recall': test_recall
    }

    print(f"  {name}")
    print(f"     Val  → Acc:{val_acc:.4f} | F1:{val_f1:.4f} | AUC:{val_auc:.4f} | Recall:{val_recall:.4f}")
    print(f"     Test → Acc:{test_acc:.4f} | F1:{test_f1:.4f} | AUC:{test_auc:.4f} | Recall:{test_recall:.4f}\n")


In [ ]:
model_colors = {
    'Random Forest' : '#2196F3',
    'XGBoost'       : '#FF9800',
    'SVM'           : '#9C27B0'
}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_val, res['pipeline'].predict(X_val))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Benign','Pathogenic'],
                yticklabels=['Benign','Pathogenic'])
    ax.set_title(f'{name}\nAcc:{res["val_acc"]:.3f}')
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')

fig.suptitle('Confusion Matrix — Validation (All Models)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, res) in zip(axes, results.items()):
    cm_t = confusion_matrix(y_test, res['pipeline'].predict(X_test))
    sns.heatmap(cm_t, annot=True, fmt='d', cmap='Oranges', ax=ax,
                xticklabels=['Benign','Pathogenic'],
                yticklabels=['Benign','Pathogenic'])
    ax.set_title(f'{name}\nAcc:{res["test_acc"]:.3f}')
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')

fig.suptitle('Confusion Matrix — Test (All Models)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for name, res in results.items():
    color      = model_colors[name]
    y_val_prob = res['pipeline'].predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_val_prob)
    auc_val     = roc_auc_score(y_val, y_val_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc_val:.3f})')
    ax.fill_between(fpr, tpr, alpha=0.05, color=color)

ax.plot([0,1],[0,1],'k--', label='Random Chance')
ax.set_xlabel('FPR')
ax.set_ylabel('TPR')
ax.set_title('ROC Curve Validation — All Models')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for name, res in results.items():
    color           = model_colors[name]
    y_test_prob     = res['pipeline'].predict_proba(X_test)[:, 1]
    fpr_t, tpr_t, _ = roc_curve(y_test, y_test_prob)
    auc_test        = roc_auc_score(y_test, y_test_prob)
    ax.plot(fpr_t, tpr_t, color=color, lw=2, label=f'{name} (AUC={auc_test:.3f})')
    ax.fill_between(fpr_t, tpr_t, alpha=0.05, color=color)

ax.plot([0,1],[0,1],'k--', label='Random Chance')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curve — Test (All Models)')
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

In [ ]:
metrics = ['Accuracy', 'F1', 'AUC', 'Recall']
x       = np.arange(len(metrics))
width   = 0.25  # narrower to fit 3 models

fig, ax = plt.subplots(figsize=(10, 5))

for i, (name, res) in enumerate(results.items()):
    color    = model_colors[name]
    val_vals = [res['val_acc'], res['val_f1'], res['val_auc'], res['val_recall']]
    offset   = (i - 1) * width
    bars     = ax.bar(x + offset, val_vals, width, label=name,
                      color=color, alpha=0.8, edgecolor='black')
    for bar, v in zip(bars, val_vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                f'{v:.2f}', ha='center', fontsize=7, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.15); ax.legend(); ax.grid(axis='y')
ax.set_title('Validation Performance — All Models')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, res) in zip(axes, results.items()):
    color     = model_colors[name]
    estimator = res['pipeline'].named_steps['model']
    if hasattr(estimator, 'feature_importances_'):
        importances = estimator.feature_importances_
        feat_names  = X.columns.tolist()
        indices     = np.argsort(importances)[::-1][:12]
        ax.bar(range(12), importances[indices], color=color, edgecolor='black')
        ax.set_xticks(range(12))
        ax.set_xticklabels([feat_names[i] for i in indices],
                           rotation=45, ha='right', fontsize=7)
        ax.set_ylabel('Importance'); ax.grid(axis='y')
    else:
        ax.text(0.5, 0.5, 'Not available\nfor SVM',
                ha='center', va='center', transform=ax.transAxes, fontsize=12)
    ax.set_title(f'{name} — Top 12 Features')

fig.suptitle('Feature Importance — All Models', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, res) in zip(axes, results.items()):
    report    = classification_report(y_test, res['pipeline'].predict(X_test),
                    target_names=['Benign','Pathogenic'], output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    report_df = report_df.drop(['accuracy','macro avg','weighted avg'], errors='ignore')
    report_df = report_df[['precision','recall','f1-score']]
    sns.heatmap(report_df, annot=True, fmt='.3f', cmap='YlOrRd',
                ax=ax, vmin=0, vmax=1)
    ax.set_title(f'{name}')

fig.suptitle('Classification Report — Test (All Models)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
model_names = list(results.keys())
colors_bar  = ['#2196F3', '#FF9800', '#9C27B0']

# ROC Curves — Validation vs Test
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for (name, res), color in zip(results.items(), colors_bar):
    y_prob          = res['pipeline'].predict_proba(X_val)[:, 1]
    fpr, tpr, _     = roc_curve(y_val, y_prob)
    auc             = roc_auc_score(y_val, y_prob)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')

    y_prob_t        = res['pipeline'].predict_proba(X_test)[:, 1]
    fpr_t, tpr_t, _ = roc_curve(y_test, y_prob_t)
    auc_t           = roc_auc_score(y_test, y_prob_t)
    axes[1].plot(fpr_t, tpr_t, color=color, lw=2, label=f'{name} (AUC={auc_t:.3f})')

for ax, title in zip(axes, ['ROC — Validation', 'ROC — Test']):
    ax.plot([0,1],[0,1],'k--', label='Random')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(title)
    ax.legend(); ax.grid(True)

plt.suptitle('ROC Curves — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('all_models_roc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for row, (split, title_prefix) in enumerate(zip(
        ['val', 'test'], ['Validation', 'Test'])):
    for col, (metric, ylabel) in enumerate(zip(
            [f'{split}_acc', f'{split}_f1', f'{split}_auc'],
            ['Accuracy', 'F1 Score', 'AUC'])):
        values = [results[n][metric] for n in model_names]
        bars   = axes[row][col].bar(model_names, values,
                                    color=colors_bar, edgecolor='black')
        axes[row][col].set_ylim(0, 1.15)
        axes[row][col].set_title(f'{title_prefix} — {ylabel}')
        axes[row][col].grid(axis='y')
        axes[row][col].tick_params(axis='x', rotation=15)
        for bar, val in zip(bars, values):
            axes[row][col].text(
                bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', fontweight='bold')

plt.suptitle('All Models — Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('all_models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("\n" + "=" * 72)
print(f"{'Model':<18} {'ValAcc':>8} {'ValF1':>7} {'ValAUC':>8} "
      f"{'TestAcc':>9} {'TestF1':>8} {'TestAUC':>9}")
print("=" * 72)

for name in model_names:
    r = results[name]
    print(f"{name:<18} {r['val_acc']:>8.4f} {r['val_f1']:>7.4f} "
          f"{r['val_auc']:>8.4f} {r['test_acc']:>9.4f} "
          f"{r['test_f1']:>8.4f} {r['test_auc']:>9.4f}")

print("=" * 72)

best_name  = max(results, key=lambda k: results[k]['test_f1'])
best_model = results[best_name]['pipeline']
print(f"\n  Best Model : {best_name}")

In [ ]:
#output prediction code for test data

print("=" * 75)
print("          FULL TEST SET PREDICTIONS")
print("=" * 75)

test_predictions = {}

for name, res in results.items():
    y_prob  = res['pipeline'].predict_proba(X_test)[:, 1]
    y_pred  = res['pipeline'].predict(X_test)

    labels     = ['Pathogenic' if p == 1 else 'Benign' for p in y_pred]
    confidence = [f"{prob*100:.2f}%" for prob in y_prob]

    test_predictions[name] = {
        'y_pred'    : y_pred,
        'y_prob'    : y_prob,
        'labels'    : labels,
        'confidence': confidence
    }

    print(f"\n {name} — Predictions Done")
    print(f"   Pathogenic predicted : {sum(y_pred == 1)}")
    print(f"   Benign predicted     : {sum(y_pred == 0)}")

In [ ]:
# Get original variant info from X_test indices
pred_df = pd.DataFrame({
    'CHROM'      : df_main.loc[X_test.index, 'CHROM'].values   if 'CHROM' in df_main.columns else X_test.index,
    'POS'        : df_main.loc[X_test.index, 'POS'].values     if 'POS'   in df_main.columns else X_test.index,
    'REF'        : df_main.loc[X_test.index, 'REF'].values     if 'REF'   in df_main.columns else '-',
    'ALT'        : df_main.loc[X_test.index, 'ALT'].values     if 'ALT'   in df_main.columns else '-',
    'True_Label' : ['Pathogenic' if v == 1 else 'Benign' for v in y_test],
})

# Add each model's prediction + confidence
for name, preds in test_predictions.items():
    col = name.replace(' ', '_')
    pred_df[f'{col}_Pred']       = preds['labels']
    pred_df[f'{col}_Confidence'] = preds['confidence']

# Add agreement column (all 3 models agree?)
pred_df['All_Agree'] = (
    (pred_df['Random_Forest_Pred'] == pred_df['XGBoost_Pred']) &
    (pred_df['XGBoost_Pred']       == pred_df['SVM_Pred'])
)

print(f" Prediction DataFrame shape : {pred_df.shape}")
print(f"   All 3 models agree         : {pred_df['All_Agree'].sum()} / {len(pred_df)}")
print(f"   Disagreements              : {(~pred_df['All_Agree']).sum()} / {len(pred_df)}")
pred_df.head(10)

In [ ]:
print("=" * 60)
print("        CORRECT vs WRONG PREDICTIONS")
print("=" * 60)

for name in results.keys():
    col      = name.replace(' ', '_')
    correct  = (pred_df[f'{col}_Pred'] == pred_df['True_Label']).sum()
    wrong    = len(pred_df) - correct
    total    = len(pred_df)

    print(f"\n  {name}")
    print(f"   Correct : {correct:>6} / {total} ({correct/total*100:.2f}%)")
    print(f"   Wrong   : {wrong:>6} / {total} ({wrong/total*100:.2f}%)")

print("=" * 60)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, preds) in zip(axes, test_predictions.items()):
    color     = model_colors[name]
    n_path    = sum(p == 1 for p in preds['y_pred'])
    n_benign  = sum(p == 0 for p in preds['y_pred'])

    bars = ax.bar(['Benign', 'Pathogenic'],
                  [n_benign, n_path],
                  color=['#4CAF50', '#F44336'],
                  edgecolor='black')
    ax.set_title(f'{name}\nTotal: {len(preds["y_pred"])}')
    ax.set_ylabel('Count')
    ax.grid(axis='y')
    for bar, val in zip(bars, [n_benign, n_path]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 10,
                f'{val}\n({val/len(preds["y_pred"])*100:.1f}%)',
                ha='center', fontweight='bold')

fig.suptitle('Prediction Distribution — Test Set (All Models)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, preds) in zip(axes, test_predictions.items()):
    color = model_colors[name]

    # Split confidence by true label
    prob_benign     = preds['y_prob'][y_test == 0]
    prob_pathogenic = preds['y_prob'][y_test == 1]

    ax.hist(prob_benign,     bins=40, alpha=0.6,
            color='#4CAF50', label='True Benign',     edgecolor='black')
    ax.hist(prob_pathogenic, bins=40, alpha=0.6,
            color='#F44336', label='True Pathogenic', edgecolor='black')
    ax.axvline(x=0.5, color='black', linestyle='--', label='Threshold 0.5')
    ax.set_xlabel('Predicted Probability (Pathogenic)')
    ax.set_ylabel('Count')
    ax.set_title(f'{name}')
    ax.legend(fontsize=8); ax.grid(True)

fig.suptitle('Confidence Score Distribution — Test Set',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
import os

SAVE_PATH = "/home/dsce-bt/Desktop/predictions_saved_testdata_csvformat"
os.makedirs(SAVE_PATH, exist_ok=True)

# Save CSV
full = os.path.join(SAVE_PATH, 'test_predictions.csv')
pred_df.to_csv(full, index=False)
print(f" Saved → {full}")

# Preview
print(f"\n Preview of saved file:")
print(pred_df[['True_Label',
               'Random_Forest_Pred', 'Random_Forest_Confidence',
               'XGBoost_Pred',       'XGBoost_Confidence',
               'SVM_Pred',           'SVM_Confidence',
               'All_Agree']].head(10).to_string())

In [ ]:
disagreements = pred_df[~pred_df['All_Agree']].copy()

print(f"=" * 70)
print(f"   DISAGREEMENT ANALYSIS — {len(disagreements)} variants")
print(f"=" * 70)
print(f"\n{'True':<15} {'RF':<15} {'XGB':<15} {'SVM':<15} {'RF_Conf':<12} {'XGB_Conf':<12} {'SVM_Conf'}")
print("-" * 70)

for _, row in disagreements.head(20).iterrows():
    print(f"{row['True_Label']:<15} "
          f"{row['Random_Forest_Pred']:<15} "
          f"{row['XGBoost_Pred']:<15} "
          f"{row['SVM_Pred']:<15} "
          f"{row['Random_Forest_Confidence']:<12} "
          f"{row['XGBoost_Confidence']:<12} "
          f"{row['SVM_Confidence']}")

print("=" * 70)
print(f"\n  Disagreement breakdown:")
print(f"  RF  vs XGB differ : {(pred_df['Random_Forest_Pred'] != pred_df['XGBoost_Pred']).sum()}")
print(f"  XGB vs SVM differ : {(pred_df['XGBoost_Pred']       != pred_df['SVM_Pred']).sum()}")
print(f"  RF  vs SVM differ : {(pred_df['Random_Forest_Pred'] != pred_df['SVM_Pred']).sum()}")